# 02. Feature Engineering
초기 100 사이클에서 피처를 만들고, 배치별로 수명과의 관계가 일관된지 확인한다.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from config import DATA_DIR, RESULT_DIR
print(DATA_DIR)

/Users/parksihyun/Desktop/ess-battery-project/data


In [2]:
import pandas as pd
from preprocess import load_cells, clean_cells, first_valid_idx
from features import build_feature_table, FEATURE_SETS
cells, report = clean_cells(load_cells())

# 배치별 '첫 유효 사이클' index 확인 (배치마다 다르면 index를 그대로 비교할 수 없음)
report.groupby('batch')['first_valid_idx'].value_counts()

batch  first_valid_idx
b1     1                  46
b2     0                  47
b3     0                  46
Name: count, dtype: int64

In [3]:
df = build_feature_table(cells)
df.to_csv(os.path.join(RESULT_DIR, 'features.csv'), index=False)
print(df.groupby('batch').size())
df.head()

batch
b1    36
b2    39
b3    44
dtype: int64


,cell,batch,policy,cycle_life,dQ_logvar,dQ_min,dQ_mean,dQ_skew,dQ_kurt,QD_c2,QD_max_minus_c2,QD_slope_2_100,IR_c2,IR_diff,Tavg_mean,chargetime_mean,C_avg,y
0,b1c5,b1,4.4C(80%)-4.4C,1074.0,-4.155832,-1.583636,-1.937981,-0.141296,-1.192363,1.077600,0.004348,0.000005,0.016396,-0.000083,30.919684,10.950920,4.400000,3.031004
1,b1c6,b1,4.8C(80%)-4.8C,636.0,-3.771283,-1.414229,-1.791400,-0.393456,-1.210020,1.078405,0.003681,-0.000011,0.016977,-0.000309,32.271324,10.058975,4.800000,2.803457
2,b1c7,b1,4.8C(80%)-4.8C,870.0,-3.821834,-1.428857,-1.784616,-0.308700,-1.112408,1.095954,0.001906,-0.000012,0.016268,0.000006,31.988196,10.042062,4.800000,2.939519
3,b1c9,b1,5.4C(40%)-3.6C,1054.0,-3.978645,-1.528094,-1.913511,-0.459565,-1.204904,1.084230,0.004633,0.000015,0.016929,0.000024,32.641869,11.207927,4.320000,3.022841
4,b1c11,b1,5.4C(50%)-3C,788.0,-4.099072,-1.598897,-2.045593,-0.605515,-1.183651,1.055879,0.005354,0.000017,0.016582,-0.000312,32.757411,11.685500,4.153846,2.896526


## 배치별 상관 : 부호가 일관된 피처만 채택

In [4]:
corr = pd.DataFrame({b: g[FEATURE_SETS['full']].corrwith(g['y']) for b, g in df.groupby('batch')})
corr['all'] = df[FEATURE_SETS['full']].corrwith(df['y'])
corr.round(2).sort_values('all', key=abs, ascending=False)

,b1,b2,b3,all
dQ_logvar,-0.84,-0.92,-0.76,-0.90
dQ_min,-0.55,-0.90,-0.74,-0.84
dQ_mean,-0.77,-0.87,-0.65,-0.81
QD_c2,0.15,-0.19,0.17,-0.59
QD_max_minus_c2,0.35,-0.44,0.01,-0.48
Tavg_mean,-0.16,0.39,-0.01,0.37
dQ_kurt,0.28,0.65,0.34,0.32
chargetime_mean,0.58,-0.94,0.60,0.23
IR_diff,0.13,0.38,-0.46,0.21
C_avg,-0.59,0.35,-0.02,-0.15
